# CleitonForge vs Qiskit — Neutral Benchmark Comparison

**Author:** Cleiton Augusto Correa Bezerra  
**Project:** [CleitonForge](https://github.com/cleitonaugusto/CleitonForge) | [PyPI](https://pypi.org/project/cleitonforge/)  
**License:** Apache 2.0

---

This notebook demonstrates **CleitonForge** as a neutral benchmarking layer for quantum circuit simulators.

We run the **same circuits** on both CleitonForge and Qiskit and compare:
- Measurement results (counts, probabilities)
- Circuit metrics (depth, gate count)
- Execution time
- Cross-backend fidelity (statevector vs QuantRS2)

CleitonForge does **not** compete with Qiskit — it is the neutral arbiter that measures any simulator without favoring any backend.

## Setup

In [1]:
# pip install cleitonforge qiskit qiskit-aer
import time
import math
import numpy as np
import cleitonforge as cf
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
import qiskit.qasm2 as qasm2

import qiskit, qiskit_aer
print(f'cleitonforge : installed')
print(f'qiskit       : {qiskit.__version__}')
print(f'qiskit-aer   : {qiskit_aer.__version__}')

SHOTS = 4096
SEED  = 42
sim   = AerSimulator(seed_simulator=SEED)

cleitonforge : installed
qiskit       : 2.5.0
qiskit-aer   : 0.17.2


---
## 1. Bell State — |Φ⁺⟩ = (|00⟩ + |11⟩) / √2

The simplest entangled state. Expected: 50% |00⟩, 50% |11⟩.

In [2]:
# ── CleitonForge ─────────────────────────────────────────────────────────
t0 = time.perf_counter()
cf_bell = cf.Circuit(2)
cf_bell.h(0); cf_bell.cx(0, 1)
cf_result = cf.run(cf_bell, shots=SHOTS, seed=SEED)
cf_time = (time.perf_counter() - t0) * 1000

# ── Qiskit / Aer ─────────────────────────────────────────────────────────
t0 = time.perf_counter()
qk_bell = QuantumCircuit(2, 2)
qk_bell.h(0); qk_bell.cx(0, 1)
qk_bell.measure([0, 1], [0, 1])
qk_counts = sim.run(qk_bell, shots=SHOTS).result().get_counts()
qk_time = (time.perf_counter() - t0) * 1000

print('Bell State — counts')
print(f'  CleitonForge : {dict(sorted(cf_result.counts.items()))}  ({cf_time:.1f} ms)')
print(f'  Qiskit/Aer   : {dict(sorted(qk_counts.items()))}  ({qk_time:.1f} ms)')

# Exact statevector
cf_sv = cf.run(cf_bell, shots=0).statevector
qk_bell_sv = QuantumCircuit(2); qk_bell_sv.h(0); qk_bell_sv.cx(0, 1)
qk_sv = Statevector(qk_bell_sv).data
print(f'\nStatevector CleitonForge : {[complex(round(r,4), round(i,4)) for r,i in cf_sv]}')
print(f'Statevector Qiskit       : {[complex(round(v.real,4), round(v.imag,4)) for v in qk_sv]}')

Bell State — counts
  CleitonForge : {'00': 2054, '11': 2042}  (0.6 ms)
  Qiskit/Aer   : {'00': 2009, '11': 2087}  (7.4 ms)

Statevector CleitonForge : [(0.7071+0j), 0j, 0j, (0.7071+0j)]
Statevector Qiskit       : [(0.7071+0j), 0j, 0j, (0.7071+0j)]


---
## 2. Circuit Metrics — canonical IR vs Qiskit transpiler

CleitonForge computes metrics over the **canonical IR** — independent of any backend.
This guarantees fair comparison regardless of which simulator runs the circuit.

In [3]:
# GHZ state (3 qubits)
cf_ghz = cf.Circuit(3)
cf_ghz.h(0); cf_ghz.cx(0, 1); cf_ghz.cx(1, 2)
qk_ghz = QuantumCircuit(3)
qk_ghz.h(0); qk_ghz.cx(0, 1); qk_ghz.cx(1, 2)

# QFT 3 qubits
cf_qft = cf.Circuit(3)
cf_qft.h(0)
cf_qft.cp(np.pi/2, 0, 1); cf_qft.cp(np.pi/4, 0, 2)
cf_qft.h(1); cf_qft.cp(np.pi/2, 1, 2)
cf_qft.h(2); cf_qft.swap(0, 2)

qk_qft = QuantumCircuit(3)
qk_qft.h(0)
qk_qft.cp(np.pi/2, 0, 1); qk_qft.cp(np.pi/4, 0, 2)
qk_qft.h(1); qk_qft.cp(np.pi/2, 1, 2)
qk_qft.h(2); qk_qft.swap(0, 2)

basis = ['h', 'cx', 'cp', 'swap', 'x', 'rz', 'sx']
circuits_meta = [
    ('Bell (2q)', cf_bell, QuantumCircuit(2)),
    ('GHZ (3q)',  cf_ghz,  qk_ghz),
    ('QFT (3q)',  cf_qft,  qk_qft),
]

print(f'{"Circuit":<12} {"Tool":<14} {"Qubits":>7} {"Gates":>7} {"Depth":>7}')
print('-' * 52)
for name, cf_c, qk_c in circuits_meta:
    qk_t = transpile(qk_c, basis_gates=basis)
    print(f'{name:<12} {"CleitonForge":<14} {cf_c.num_qubits:>7} {cf_c.gate_count:>7} {cf_c.depth:>7}')
    print(f'{"":<12} {"Qiskit":<14} {qk_t.num_qubits:>7} {qk_t.size():>7} {qk_t.depth():>7}')
    print()

Circuit      Tool            Qubits   Gates   Depth
----------------------------------------------------
Bell (2q)    CleitonForge         2       2       2
             Qiskit               2       0       0

GHZ (3q)     CleitonForge         3       3       3
             Qiskit               3       3       3

QFT (3q)     CleitonForge         3       7       6
             Qiskit               3       6       5



---
## 3. Grover's Algorithm — 3 qubits, oracle |101⟩

Grover search with oracle marking |101⟩. After 2 iterations the marked state  
should be measured with probability ≈ 94.5% (theoretical: sin²(5θ) ≈ 94.8%).

**Oracle**: CCZ is decomposed as `H q2 → CCX(q0,q1,q2) → H q2` (phase kickback).  
**Diffuser**: same CCZ decomposition for inversion about the mean.

In [4]:
def build_cf_grover():
    """3-qubit Grover targeting |101⟩, 2 iterations. CCZ = H·CCX·H."""
    c = cf.Circuit(3)
    c.h(0); c.h(1); c.h(2)          # uniform superposition
    for _ in range(2):
        # Oracle: phase-flip |101⟩ → map to |111⟩ then CCZ then undo
        c.x(1)
        c.h(2); c.ccx(0, 1, 2); c.h(2)   # CCZ via phase kickback
        c.x(1)
        # Diffuser: inversion about mean
        c.h(0); c.h(1); c.h(2)
        c.x(0); c.x(1); c.x(2)
        c.h(2); c.ccx(0, 1, 2); c.h(2)   # CCZ
        c.x(0); c.x(1); c.x(2)
        c.h(0); c.h(1); c.h(2)
    return c

def build_qk_grover():
    qc = QuantumCircuit(3, 3)
    qc.h([0, 1, 2])
    for _ in range(2):
        qc.x(1)
        qc.h(2); qc.ccx(0, 1, 2); qc.h(2)
        qc.x(1)
        qc.h([0,1,2]); qc.x([0,1,2])
        qc.h(2); qc.ccx(0, 1, 2); qc.h(2)
        qc.x([0,1,2]); qc.h([0,1,2])
    qc.measure([0,1,2],[0,1,2])
    return qc

t0 = time.perf_counter()
grover_cf = build_cf_grover()
gr_result = cf.run(grover_cf, shots=SHOTS, seed=SEED)
cf_grover_time = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
qk_gr_counts = sim.run(build_qk_grover(), shots=SHOTS).result().get_counts()
qk_grover_time = (time.perf_counter() - t0) * 1000

theoretical = math.sin((2*2+1) * math.asin(1/math.sqrt(8)))**2
cf_prob = gr_result.counts.get('101', 0) / SHOTS
qk_prob = qk_gr_counts.get('101', 0) / SHOTS

print('Grover 3q — target |101⟩')
print(f'  Theoretical P(|101⟩) : {theoretical:.4f}  ({theoretical*100:.1f}%)')
print(f'  CleitonForge         : {cf_prob:.4f}  ({cf_prob*100:.1f}%)  [{cf_grover_time:.1f} ms]')
print(f'  Qiskit/Aer           : {qk_prob:.4f}  ({qk_prob*100:.1f}%)  [{qk_grover_time:.1f} ms]')
print(f'\n  Top-3 CleitonForge : {gr_result.top_states(3)}')
print(f'  Top-3 Qiskit       : {sorted(qk_gr_counts.items(), key=lambda x: -x[1])[:3]}')

Grover 3q — target |101⟩
  Theoretical P(|101⟩) : 0.9453  (94.5%)
  CleitonForge         : 0.9463  (94.6%)  [0.4 ms]
  Qiskit/Aer           : 0.9495  (94.9%)  [5.6 ms]

  Top-3 CleitonForge : [('101', 0.9453125000000023), ('100', 0.007812500000000043), ('000', 0.007812500000000023)]
  Top-3 Qiskit       : [('101', 3889), ('110', 41), ('000', 33)]


---
## 4. Cross-Backend Fidelity — statevector vs QuantRS2

CleitonForge runs the **same circuit** on two independent backends and checks
that results are equivalent. This is the core value proposition: neutral validation.

In [5]:
to_check = [
    ('Bell state', cf_bell),
    ('GHZ state',  cf_ghz),
    ('QFT 3q',     cf_qft),
    ('Grover 3q',  grover_cf),
]

print(f'{"Circuit":<14} {"Backend A":>12} {"Backend B":>12} {"Fidelity":>12} {"Match":>6}')
print('-' * 62)

for name, circuit in to_check:
    sv_a = cf.run(circuit, shots=0, backend='statevector').statevector
    sv_b = cf.run(circuit, shots=0, backend='quantrs2').statevector
    dot      = sum(a[0]*b[0] + a[1]*b[1] for a, b in zip(sv_a, sv_b))
    fidelity = dot ** 2
    match = '✅' if abs(fidelity - 1.0) < 1e-6 else '❌'
    print(f'{name:<14} {"statevector":>12} {"quantrs2":>12} {fidelity:>12.8f} {match:>6}')

Circuit           Backend A    Backend B     Fidelity  Match
--------------------------------------------------------------
Bell state      statevector     quantrs2   1.00000000      ✅
GHZ state       statevector     quantrs2   1.00000000      ✅
QFT 3q          statevector     quantrs2   1.00000000      ✅
Grover 3q       statevector     quantrs2   1.00000000      ✅


---
## 5. Run a Qiskit circuit on CleitonForge via QASM

Export any Qiskit circuit to QASM and run it directly on CleitonForge — zero Rust knowledge required.

In [6]:
qasm_str = qasm2.dumps(qk_ghz)
print('QASM exported from Qiskit:')
print(qasm_str)

# CleitonForge handles stdgates.inc internally — strip the include line
cleaned = '\n'.join(l for l in qasm_str.splitlines() if not l.strip().startswith('include'))
result  = cf.run_qasm(cleaned, shots=SHOTS, seed=SEED)

print(f'CleitonForge result from Qiskit QASM : {result.counts}')
print(f'Top states                           : {result.top_states(2)}')

QASM exported from Qiskit:
OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
h q[0];
cx q[0],q[1];
cx q[1],q[2];
CleitonForge result from Qiskit QASM : {'111': 2042, '000': 2054}
Top states                           : [('000', 0.5000000000000001), ('111', 0.5000000000000001)]


---
## Summary

| Metric | CleitonForge | Qiskit/Aer |
|--------|-------------|------------|
| Language | Rust (Python bindings) | Python |
| Backends | statevector, QuantRS2 | Aer (statevector, GPU…) |
| Input formats | OpenQASM 2/3, Python API | Python API, QASM |
| Neutral arbiter | ✅ | ❌ (IBM product) |
| Cross-backend fidelity check | ✅ | ❌ |
| Canonical IR metrics | ✅ | ❌ |
| Open source | ✅ Apache 2.0 | ✅ Apache 2.0 |

**CleitonForge is not a Qiskit replacement.**  
It is the neutral layer researchers use to validate that multiple simulators agree — and to benchmark them fairly.

---

**Author:** Cleiton Augusto Correa Bezerra  
**GitHub:** https://github.com/cleitonaugusto/CleitonForge  
**PyPI:** https://pypi.org/project/cleitonforge/  
**License:** Apache 2.0